<a href="https://colab.research.google.com/github/Adigozalovh/DeepLearning/blob/main/UNet_Medical_Image_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets
from torch.utils.data import Dataset, DataLoader

In [2]:
class Conv3(nn.Module):
  def __init__(self, n_features, out_channel):
    super().__init__()
    self.calc = nn.Sequential(
        nn.Conv2d(n_features, out_channel, 3, 1, 1),
        nn.ReLU(),
        nn.Conv2d(out_channel, out_channel, 3, 1, 1),
        nn.ReLU()
    )

  def forward(self, X):
    return self.calc(X)

In [3]:
class UpConv(nn.Module):
  def __init__(self, n_features, out_channel):
    super().__init__()

    self.upconv = nn.ConvTranspose2d(n_features, out_channel, kernel_size=2, stride=2)

  def forward(self, X):
    return self.upconv(X)

In [4]:
class UNet(nn.Module):
  def __init__(self, n_features, out_channel):
    super().__init__()

    self.max_pool = nn.MaxPool2d(2, 2)

    # DownConvs
    self.DownConv1 = Conv3(n_features, 64)
    self.DownConv2 = Conv3(64, 128)
    self.DownConv3 = Conv3(128, 256)
    self.DownConv4 = Conv3(256, 512)

    # Bottleneck
    self.Bottleneck = Conv3(512, 1024)

    # UpConv
    self.UpConv1 = UpConv(1024, 512)
    self.UpConv2 = UpConv(512, 256)
    self.UpConv3 = UpConv(256, 128)
    self.UpConv4 = UpConv(128, 64)

    # Expansion
    self.Expansion1 = Conv3(1024, 512)
    self.Expansion2 = Conv3(512, 256)
    self.Expansion3 = Conv3(256, 128)
    self.Expansion4 = Conv3(128, 64)

    # output
    self.output = nn.Conv2d(64, out_channel, 1, stride=1)

  def forward(self, X):
    # Down
    X = self.DownConv1(X)
    skip1 = X
    X = self.max_pool(X)
    X = self.DownConv2(X)
    skip2 = X
    X = self.max_pool(X)
    X = self.DownConv3(X)
    skip3 = X
    X = self.max_pool(X)
    X = self.DownConv4(X)
    skip4 = X
    X = self.max_pool(X)
    X = self.Bottleneck(X)

    # Up
    X = self.UpConv1(X)
    X = torch.cat([X, skip4], dim=1)
    X = self.Expansion1(X)
    X = self.UpConv2(X)
    X = torch.cat([X, skip3], dim=1)
    X = self.Expansion2(X)
    X = self.UpConv3(X)
    X = torch.cat([X, skip2], dim=1)
    X = self.Expansion3(X)
    X = self.UpConv4(X)
    X = torch.cat([X, skip1], dim=1)
    X = self.Expansion4(X)

    X = self.output(X)

    return X

In [5]:
def calculate_dice_score(preds, labels, eps=1e-8):
  proba = torch.sigmoid(preds)
  preds = (proba > 0.5).float()

  preds = preds.view(-1)
  labels = labels.view(-1)

  intersection = (preds * labels).sum()
  dice_score = (2.0 * intersection + eps) / (preds.sum() + labels.sum() + eps)

  return dice_score

In [6]:
def eval(model, criterion, valid_loader, device):
  valid_losses = 0.
  valid_dice_scores = 0.
  model.eval()

  with torch.no_grad():
    for image, mask in valid_loader:
      image, mask = image.to(device), mask.to(device)
      preds = model(image)
      loss = criterion(preds, mask)
      dice_score = calculate_dice_score(preds, mask)
      valid_losses += loss.item()
      valid_dice_scores += dice_score.item()

    avg_loss = valid_losses / len(valid_loader)
    avg_dice_score = valid_dice_scores / len(valid_loader)

  return avg_loss, avg_dice_score


In [7]:
def train(model, criterion, optimizer, train_loader,
          valid_loader, scheduler, n_epochs, device):
  history = {
      'train_loss':[],
      'train_metric':[],
      'valid_loss':[],
      'valid_metric':[],
      'lr':[]
  }

  best_dice_score = 0.
  for epoch in range(n_epochs):
    train_losses = 0.
    train_dice_scores = 0.
    model.train()
    current_lr = optimizer.param_groups[0]['lr']
    for image, mask in train_loader:
      image, mask = image.to(device), mask.to(device)
      with torch.cuda.amp.autocast(enabled=(device=='cuda')):
        preds = model(image)
        loss = criterion(preds, mask)
        dice_score = calculate_dice_score(preds, mask)
        train_losses += loss.item()
        train_dice_scores += dice_score.item()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    avg_train_loss = train_losses / len(train_loader)
    avg_dice_score = train_dice_scores / len(train_loader)
    avg_valid_loss, avg_valid_dice_score = eval(model, criterion,
                                                valid_loader, device)

    if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
      scheduler.step(avg_valid_loss)
    else:
      scheduler.step()

    if avg_dice_score > best_dice_score:
      best_dice_score = avg_dice_score

    history['train_loss'].append(avg_train_loss)
    history['train_metric'].append(avg_dice_score)
    history['valid_loss'].append(avg_valid_loss)
    history['valid_metric'].append(avg_valid_dice_score)
    history['lr'].append(current_lr)

    print({f'Epoch: {epoch+1}/{n_epochs}| Train Loss: {avg_train_loss:^5}| Train Dice Score: {avg_dice_score:^5}| Val Loss: {avg_valid_loss:^5}| Val Dice Score: {avg_valid_dice_score:^5}| LR: {current_lr}'})

  return history

In [8]:
#!/bin/bash
!curl -L -o isbi-2012-challenge.zip https://www.kaggle.com/api/v1/datasets/download/hamzamohiuddin/isbi-2012-challenge && unzip isbi-2012-challenge.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 12.5M  100 12.5M    0     0  3133k      0  0:00:04  0:00:04 --:--:-- 3415k
Archive:  isbi-2012-challenge.zip
  inflating: unmodified-data/test/imgs/frame_0001.png  
  inflating: unmodified-data/test/imgs/frame_0002.png  
  inflating: unmodified-data/test/imgs/frame_0003.png  
  inflating: unmodified-data/test/imgs/frame_0004.png  
  inflating: unmodified-data/test/imgs/frame_0005.png  
  inflating: unmodified-data/test/imgs/frame_0006.png  
  inflating: unmodified-data/test/imgs/frame_0007.png  
  inflating: unmodified-data/test/imgs/frame_0008.png  
  inflating: unmodified-data/test/imgs/frame_0009.png  
  inflating: unmodified-data/test/imgs/frame_0010.png  
  inflating: unmodified-data/test/imgs/frame_0011.png  
  inflating: unmodified-data/test/

In [9]:
import os
import glob
from PIL import Image

In [10]:
class SegmentationDataset(Dataset):
  def __init__(self, image_path, mask_path):
    self.image_files = sorted(glob.glob(os.path.join(image_path, '*.png')))
    self.mask_files = sorted(glob.glob(os.path.join(mask_path, '*.png')))

    self.transform = transforms.Compose([
        transforms.ToTensor()
    ])

  def __len__(self):
    return len(self.image_files)

  def __getitem__(self, idx):
    image = Image.open(self.image_files[idx]).convert('L')
    mask = Image.open(self.mask_files[idx]).convert('L')

    return self.transform(image), self.transform(mask)

In [11]:
train_data = SegmentationDataset(image_path = '/content/unmodified-data/train/imgs',
                                 mask_path='/content/unmodified-data/train/labels')
val_data = SegmentationDataset(image_path = '/content/unmodified-data/test/imgs',
                                 mask_path='/content/unmodified-data/test/labels')


In [12]:
train_loader, valid_loader = DataLoader(train_data, batch_size=1, shuffle=True), DataLoader(val_data, batch_size=1)

In [13]:
next(iter(train_loader))[0].shape

torch.Size([1, 1, 512, 512])

In [14]:
if torch.cuda.is_available():
  device='cuda'
elif torch.backends.mps.is_available():
  device='mps'
else:
  device='cpu'

In [15]:
model = UNet(1, 1)
model.to(device)
criterion = nn.BCEWithLogitsLoss()
learning_rate = 1e-3
n_epochs=10
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

history = train(model, criterion, optimizer, train_loader, valid_loader, scheduler, n_epochs, device)

/tmp/ipykernel_4454/874928795.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device=='cuda')):


{'Epoch: 1/10| Train Loss: 0.46110853950182595| Train Dice Score: 0.8765351633230846| Val Loss: 0.4524053911368052| Val Dice Score: 0.8394041419029236| LR: 0.001'}
{'Epoch: 2/10| Train Loss: 0.6945597132047018| Train Dice Score: 0.8215678073310604| Val Loss: 0.48035600980122883| Val Dice Score: 0.8322524050871531| LR: 0.001'}
{'Epoch: 3/10| Train Loss: 0.36264457205931344| Train Dice Score: 0.8796926101048788| Val Loss: 0.4427505264679591| Val Dice Score: 0.8501004417737325| LR: 0.001'}
{'Epoch: 4/10| Train Loss: 0.34955763320128125| Train Dice Score: 0.8853557924429576| Val Loss: 0.4333597222963969| Val Dice Score: 0.8593263546625773| LR: 0.001'}
{'Epoch: 5/10| Train Loss: 0.37754393021265664| Train Dice Score: 0.8844433784484863| Val Loss: 0.42099389135837556| Val Dice Score: 0.8554540117581685| LR: 0.001'}
{'Epoch: 6/10| Train Loss: 0.3439907888571421| Train Dice Score: 0.8934579869111379| Val Loss: 0.4175515333811442| Val Dice Score: 0.8580177227656046| LR: 0.001'}
{'Epoch: 7/10| T